# Pipeline complet Saxo — options actions ASML de bout en bout (V2)

Reconstruction A→Z d'une **vraie** chaîne d'options actions (ASML, Euronext Amsterdam),
**100 % offline** : on rejoue un échantillon **committé** de données Saxo réelles (delayed,
publiques) — aucune connexion broker, aucun réseau. Même moteur canonique que la prod :
events → snapshot → QC → forward → inversion IV → surface SVI → pricing/Greeks → risque →
scénarios. **Plots Plotly, thème unifié `algotrading`** (jumeau de `demo_pipeline_deribit_v2`).

> **Périmètre de l'échantillon actuel** : une **seule échéance** (2026-07-01, 45 strikes,
> calls + puts). On obtient donc smile + fit SVI 2D, pas de surface 3D (qui exige ≥ 2
> maturités). Le notebook est **générique 1..N échéances** : dès qu'un échantillon
> multi-maturités est fourni, surface 3D / term structure / violations se peuplent seules.

In [1]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

C = {
    "blue": "#2563EB",  # primary series / calls
    "teal": "#0D9488",  # puts / complementary
    "violet": "#7C3AED",  # gamma / vega
    "amber": "#D97706",  # warning / stress
    "red": "#DC2626",  # rejected / violation
    "green": "#16A34A",  # usable / pass
    "indigo": "#4F46E5",
    "slate900": "#0F172A",
    "slate600": "#475569",
    "slate400": "#94A3B8",
    "slate100": "#F1F5F9",
    "white": "#FFFFFF",
}
DISCRETE = [C["blue"], C["teal"], C["violet"], C["amber"], C["indigo"], "#0EA5E9", "#F59E0B"]
FONT = "Inter, IBM Plex Sans, -apple-system, sans-serif"

SURFACE_COLORSCALE = [
    [0.00, "#1E3A5F"],
    [0.25, "#2563EB"],
    [0.50, "#0D9488"],
    [0.75, "#D97706"],
    [1.00, "#DC2626"],
]
SURFACE_CAMERA = dict(eye=dict(x=1.6, y=-1.6, z=0.9), up=dict(x=0, y=0, z=1))
SURFACE_ASPECT = dict(x=1.5, y=1.2, z=0.7)

_axis = dict(
    showgrid=True,
    gridcolor=C["slate400"],
    gridwidth=0.5,
    zeroline=False,
    linecolor=C["slate400"],
    linewidth=1,
    ticks="outside",
    tickcolor=C["slate400"],
    tickfont=dict(size=11),
    title_font=dict(size=12, color=C["slate600"]),
)
_tmpl = go.layout.Template()
_tmpl.layout = go.Layout(
    font=dict(family=FONT, size=12, color=C["slate600"]),
    title_font=dict(family=FONT, size=15, color=C["slate900"]),
    paper_bgcolor=C["white"],
    plot_bgcolor=C["white"],
    colorway=DISCRETE,
    xaxis=_axis,
    yaxis=_axis,
    legend=dict(
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor=C["slate400"],
        borderwidth=1,
        font=dict(size=11),
        tracegroupgap=4,
    ),
    margin=dict(l=64, r=24, t=56, b=56),
    hoverlabel=dict(
        bgcolor=C["slate900"],
        font=dict(color=C["white"], size=12, family=FONT),
        bordercolor=C["slate900"],
    ),
    hovermode="closest",
)
pio.templates["algotrading"] = _tmpl
pio.templates.default = "plotly_white+algotrading"
print("Chart config loaded — template: algotrading (Plotly V2)")

Chart config loaded — template: algotrading (Plotly V2)


In [2]:
from collections import defaultdict
from decimal import Decimal
from pathlib import Path

import numpy as np
from algotrading.core.config import load_config
from algotrading.infra.forwards import build_forward_curve
from algotrading.infra.iv import solve_chain
from algotrading.infra.pricing import OptionStyle, PricingInput, PricingParams, price
from algotrading.infra.qc import filtered_snapshot, is_usable_quote, run_qc
from algotrading.infra.qc.state import QCStatus
from algotrading.infra.risk import (
    InstrumentAnalytics,
    Position,
    PositionSet,
    Scenario,
    ScenarioGrid,
    aggregate_risk,
    by_underlying,
    compute_risk_lines,
    run_grid,
)
from algotrading.infra.snapshots import build_snapshot
from algotrading.infra.storage import events_from_json
from algotrading.infra.surfaces import build_surface
from algotrading.infra.surfaces.svi import svi_total_variance
from algotrading.infra.universe.contracts import OptionContract, Right, parse_instrument_key
from algotrading.infra_saxo.flow import SaxoFlow


def _repo_root() -> Path:
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "pyproject.toml").exists() and (p / "packages").is_dir():
            return p
    raise FileNotFoundError("repo root non trouvé (pyproject.toml + packages/)")


REPO = _repo_root()
CONFIGS = REPO / "packages" / "infra" / "configs"
SYMBOL = "ASML"
SAMPLE = REPO / "packages" / "infra-saxo" / "samples" / "asml_real_2026-06-04.json"

events = events_from_json(SAMPLE.read_text(encoding="utf-8"))
AS_OF = max(e.receipt_ts for e in events)
params = SaxoFlow(infra_configs=CONFIGS).resolve_config(SYMBOL).pipeline_params

print(f"Repo        : {REPO}")
print(f"Échantillon : {SAMPLE.relative_to(REPO)}")
print(f"Sous-jacent : {SYMBOL} (actions, EUR)  |  events={len(events)}")
print(f"As-of       : {AS_OF}  |  config_hash={params.config_hash[:12]}...")

Repo        : C:\Users\Vincent\GitHub\Vincent-20-100\AlgoTrading
Échantillon : packages\infra-saxo\samples\asml_real_2026-06-04.json
Sous-jacent : ASML (actions, EUR)  |  events=810
As-of       : 2026-06-04 14:05:37.957418+00:00  |  config_hash=d6852ebe187a...


## 1 · Snapshot & contrôle qualité (QC)
`build_snapshot` agrège le dernier état par instrument/champ ; le pipeline filtre **avant**
forward/iv/surface. Aucune quote n'est écartée en silence : les raisons de rejet sont reportées.

In [3]:
snapshot = build_snapshot(events, AS_OF, params.snapshot)
print(
    f"Spot référence   : €{float(snapshot.reference_spot):,.2f} ({snapshot.reference_type.value})"
)
print(f"Options dans snap: {len(snapshot.options)}")

qc_report = run_qc(snapshot, params.qc)
clean = filtered_snapshot(snapshot, qc_report, params.qc)
n_usable = sum(1 for q in snapshot.options if is_usable_quote(q))
print(f"Options usables  : {n_usable} / {len(snapshot.options)}  ->  clean: {len(clean.options)}")

reject_reasons: dict[str, int] = {}
for v in qc_report.verdicts:
    if v.status is QCStatus.USABLE:
        continue
    for o in v.outcomes:
        if o.status is not QCStatus.USABLE and o.reason_code:
            reject_reasons[o.reason_code] = reject_reasons.get(o.reason_code, 0) + 1
if reject_reasons:
    print(f"Raisons de rejet QC  : {dict(sorted(reject_reasons.items(), key=lambda kv: -kv[1]))}")

{"ts": "2026-06-04T21:08:45.268095+00:00", "level": "INFO", "logger": "algotrading.infra.qc.engine", "message": "qc check", "check": "open_interest", "instrument_key": "OPT:ASML:OPT:20260701:C:1040:100:SAXO_236:EUR", "status": "caution", "severity": "warning", "measured": "6.000000", "reason_code": "low_open_interest", "threshold_version": "d6852ebe187a8852269cc5939c1d291717110d78d4c0c6b39040d8d3f8e21530"}


{"ts": "2026-06-04T21:08:45.269513+00:00", "level": "INFO", "logger": "algotrading.infra.qc.engine", "message": "qc check", "check": "open_interest", "instrument_key": "OPT:ASML:OPT:20260701:C:1060:100:SAXO_236:EUR", "status": "caution", "severity": "warning", "measured": "4.000000", "reason_code": "low_open_interest", "threshold_version": "d6852ebe187a8852269cc5939c1d291717110d78d4c0c6b39040d8d3f8e21530"}


{"ts": "2026-06-04T21:08:45.270630+00:00", "level": "WARNING", "logger": "algotrading.infra.qc.engine", "message": "qc check", "check": "spread_pct", "instrument_key": "OPT:ASML:OPT:20260701:C:1300:100:SAXO_236:EUR", "status": "reject", "severity": "error", "measured": "1.997655334114888628370457210", "reason_code": "spread_too_wide", "threshold_version": "d6852ebe187a8852269cc5939c1d291717110d78d4c0c6b39040d8d3f8e21530"}


{"ts": "2026-06-04T21:08:45.271544+00:00", "level": "WARNING", "logger": "algotrading.infra.qc.engine", "message": "qc check", "check": "quote_age", "instrument_key": "OPT:ASML:OPT:20260701:C:1300:100:SAXO_236:EUR", "status": "reject", "severity": "warning", "measured": "94.334495", "reason_code": "quote_too_old", "threshold_version": "d6852ebe187a8852269cc5939c1d291717110d78d4c0c6b39040d8d3f8e21530"}


{"ts": "2026-06-04T21:08:45.272121+00:00", "level": "WARNING", "logger": "algotrading.infra.qc.engine", "message": "qc check", "check": "intrinsic_sanity", "instrument_key": "OPT:ASML:OPT:20260701:C:1300:100:SAXO_236:EUR", "status": "reject", "severity": "error", "measured": "-69.600000", "reason_code": "below_intrinsic", "threshold_version": "d6852ebe187a8852269cc5939c1d291717110d78d4c0c6b39040d8d3f8e21530"}


...

{"ts": "2026-06-04T21:08:45.319600+00:00", "level": "WARNING", "logger": "algotrading.infra.qc.engine", "message": "qc check", "check": "quote_age", "instrument_key": "OPT:ASML:OPT:20260701:P:920:100:SAXO_236:EUR", "status": "reject", "severity": "warning", "measured": "181.507862", "reason_code": "quote_too_old", "threshold_version": "d6852ebe187a8852269cc5939c1d291717110d78d4c0c6b39040d8d3f8e21530"}


{"ts": "2026-06-04T21:08:45.320381+00:00", "level": "WARNING", "logger": "algotrading.infra.qc.engine", "message": "qc check", "check": "spread_pct", "instrument_key": "OPT:ASML:OPT:20260701:P:960:100:SAXO_236:EUR", "status": "reject", "severity": "error", "measured": "1.746835443037974683544303797", "reason_code": "spread_too_wide", "threshold_version": "d6852ebe187a8852269cc5939c1d291717110d78d4c0c6b39040d8d3f8e21530"}


{"ts": "2026-06-04T21:08:45.321027+00:00", "level": "WARNING", "logger": "algotrading.infra.qc.engine", "message": "qc check", "check": "quote_age", "instrument_key": "OPT:ASML:OPT:20260701:P:960:100:SAXO_236:EUR", "status": "reject", "severity": "warning", "measured": "181.507454", "reason_code": "quote_too_old", "threshold_version": "d6852ebe187a8852269cc5939c1d291717110d78d4c0c6b39040d8d3f8e21530"}


{"ts": "2026-06-04T21:08:45.321856+00:00", "level": "INFO", "logger": "algotrading.infra.qc.engine", "message": "qc summary", "underlying": "ASML", "snapshot_ts": "2026-06-04T14:05:37.957418+00:00", "threshold_version": "d6852ebe187a8852269cc5939c1d291717110d78d4c0c6b39040d8d3f8e21530", "n_usable": 40, "n_caution": 18, "n_reject": 32, "n_total": 90}


Spot référence   : €1,454.90 (mid)
Options dans snap: 90
Options usables  : 87 / 90  ->  clean: 58
Raisons de rejet QC  : {'quote_too_old': 26, 'spread_too_wide': 24, 'low_open_interest': 17, 'spread_wide': 4, 'below_intrinsic': 3, 'missing_mid': 3, 'non_positive_bid': 3}


In [4]:
strikes_all, spreads_all, usable_all = [], [], []
for q in snapshot.options:
    if q.bid is None or q.ask is None:
        continue
    c = parse_instrument_key(q.instrument_key)
    if not isinstance(c, OptionContract):
        continue
    mid = float(q.mid) if q.mid else None
    if mid and mid > 0:
        strikes_all.append(float(c.strike))
        spreads_all.append(float(q.ask - q.bid) / mid * 100)
        usable_all.append(is_usable_quote(q))

strikes_arr, spreads_arr = np.array(strikes_all), np.array(spreads_all)
usable_mask = np.array(usable_all)

fig = make_subplots(
    cols=2,
    column_widths=[0.78, 0.22],
    shared_yaxes=True,
    subplot_titles=("Spread % par strike", "Distribution"),
)
for label, color, mask in [
    ("Usable", C["green"], usable_mask),
    ("Rejeté QC", C["red"], ~usable_mask),
]:
    fig.add_trace(
        go.Scatter(
            x=strikes_arr[mask],
            y=spreads_arr[mask],
            mode="markers",
            marker=dict(color=color, size=6, opacity=0.75, line=dict(color=C["white"], width=0.5)),
            name=label,
            hovertemplate=f"<b>Strike %{{x:.0f}}</b><br>Spread: %{{y:.2f}}%<br>{label}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Histogram(
            y=spreads_arr[mask],
            nbinsy=30,
            marker_color=color,
            opacity=0.7,
            name=f"{label} dist.",
            showlegend=False,
        ),
        row=1,
        col=2,
    )
fig.add_vline(
    x=float(snapshot.reference_spot), line_dash="dash", line_color=C["slate400"], row=1, col=1
)
fig.update_layout(
    title=f"{SYMBOL} — Spread bid-ask % par strike",
    yaxis_title="Spread (%)",
    xaxis_title="Strike (EUR)",
    height=380,
    barmode="overlay",
)
fig.show()

## 2 · Courbe forward (parité put-call)
Le forward implicite par échéance, déduit de la parité call-put sur les quotes usables.

In [5]:
forward_curve = build_forward_curve(clean, params.forward)
print(f"Maturités avec forward accepté : {len(forward_curve.maturities)}")
for m in forward_curve.maturities:
    dte = int(m.maturity_years * 365)
    print(
        f"  {m.expiry}  ({dte:3d}j)  F=€{float(m.forward_price):,.2f}  "
        f"r={float(m.rate) * 100:+.2f}%  carry={float(m.implied_carry) * 100:+.2f}%  [{m.quality.value}]"
    )

Maturités avec forward accepté : 1
  2026-07-01  ( 27j)  F=€1,455.95  r=-2.53%  carry=-3.50%  [ok]


In [6]:
if not forward_curve.maturities:
    print("Aucun forward accepté — plot ignoré.")
else:
    spot_ref = float(snapshot.reference_spot)
    dtes_m = [int(m.maturity_years * 365) for m in forward_curve.maturities]
    basis = [float(m.forward_price) - spot_ref for m in forward_curve.maturities]
    rates = [float(m.rate) for m in forward_curve.maturities]
    exp_labels = [str(m.expiry) for m in forward_curve.maturities]

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=("Basis forward  (F − Spot, EUR)", "Taux de portage implicite"),
        row_heights=[0.5, 0.5],
    )
    fig.add_trace(
        go.Scatter(
            x=dtes_m,
            y=basis,
            mode="lines+markers",
            line=dict(color=C["blue"], width=2),
            marker=dict(size=7),
            customdata=exp_labels,
            hovertemplate="<b>%{customdata}</b><br>DTE: %{x}j<br>Basis: %{y:,.2f} €<extra></extra>",
            name="F − Spot",
        ),
        row=1,
        col=1,
    )
    fig.add_hline(y=0, line_dash="dash", line_color=C["slate400"], row=1, col=1)
    fig.add_trace(
        go.Scatter(
            x=dtes_m,
            y=rates,
            mode="lines+markers",
            line=dict(color=C["teal"], width=2),
            marker=dict(size=7),
            customdata=exp_labels,
            hovertemplate="<b>%{customdata}</b><br>DTE: %{x}j<br>Carry: %{y:.2%}<extra></extra>",
            name="Taux implicite",
        ),
        row=2,
        col=1,
    )
    fig.add_hline(y=0, line_dash="dash", line_color=C["slate400"], row=2, col=1)
    fig.update_yaxes(tickformat=",.0f", title_text="Basis (EUR)", row=1, col=1)
    fig.update_yaxes(tickformat=".2%", title_text="Taux", row=2, col=1)
    fig.update_xaxes(title_text="DTE (jours)", row=2, col=1)
    fig.update_layout(title=f"{SYMBOL} — Structure forward", height=500)
    fig.show()

## 3 · Inversion de la volatilité implicite
Chaque quote usable est inversée (Black-76 forward-form) en IV, positionnée en log-moneyness
`ln(K/F)`. Couleur = échéance, **trait plein = call, pointillé = put**.

In [7]:
iv_points = solve_chain(clean, forward_curve, params.iv)
n_solved = sum(1 for p in iv_points if p.status == "solved")
print(f"IvPoints : {len(iv_points)}  ->  solved: {n_solved}  unsolved: {len(iv_points) - n_solved}")
if len(iv_points) - n_solved:
    reasons: dict[str, int] = {}
    for p in iv_points:
        if p.status != "solved" and p.failure_reason:
            reasons[p.failure_reason] = reasons.get(p.failure_reason, 0) + 1
    print("  Raisons échec :", reasons)

IvPoints : 58  ->  solved: 52  unsolved: 6
  Raisons échec : {'below_intrinsic': 6}


In [8]:
solved_pts = [p for p in iv_points if p.status == "solved" and p.implied_vol is not None]
by_exp: dict = defaultdict(list)
for p in solved_pts:
    by_exp[p.expiry_date].append(p)

if not by_exp:
    print("Aucun IvPoint solved — plot ignoré.")
else:
    fig = go.Figure()
    for i, (exp, pts) in enumerate(sorted(by_exp.items())):
        color = DISCRETE[i % len(DISCRETE)]
        dte = (exp - AS_OF.date()).days
        for side, right, dash in [("Call", Right.CALL, "solid"), ("Put", Right.PUT, "dot")]:
            side_pts = sorted(
                [p for p in pts if parse_instrument_key(p.contract_key).right == right],
                key=lambda p: p.log_moneyness,
            )
            if not side_pts:
                continue
            fig.add_trace(
                go.Scatter(
                    x=[p.log_moneyness for p in side_pts],
                    y=[p.implied_vol * 100 for p in side_pts],
                    mode="lines+markers",
                    line=dict(color=color, width=1.8, dash=dash),
                    marker=dict(size=4),
                    name=f"{exp} {side}",
                    legendgroup=str(exp),
                    legendgrouptitle_text=f"{exp} ({dte}j)" if side == "Call" else None,
                    hovertemplate=f"<b>{exp} {side}</b><br>log-m: %{{x:.3f}}<br>IV: %{{y:.1f}}%<extra></extra>",
                )
            )
    fig.add_vline(x=0, line_dash="dash", line_color=C["slate400"])
    fig.update_layout(
        title=f"{SYMBOL} — Smile de volatilité implicite (skew actions)",
        xaxis_title="Log-moneyness ln(K/F)",
        yaxis_title="IV (%)",
        legend=dict(groupclick="toggleitem"),
        height=420,
    )
    fig.show()

## 4 · Surface SVI
Calibration SVI (Eq. 20) par échéance. Une maturité → une **slice** ; la **surface 3D** (et les
diagnostics calendaires) apparaissent dès qu'il y a ≥ 2 maturités.

In [9]:
solved_iv = [p for p in iv_points if p.status == "solved"]
surface_result = build_surface(solved_iv, params.surface, config_hash=params.config_hash)
print(f"Slices SVI calibrées : {len(surface_result.parameters.slices)}")
for sl in surface_result.parameters.slices:
    dte = int(sl.maturity_years * 365)
    rmse = f"RMSE={sl.rmse:.5f}" if sl.rmse is not None else "RMSE=n/a"
    warn = "  [borne!]" if sl.bound_hits else ""
    print(f"  {dte:3d}j  [{sl.method.upper():5s}]  n={sl.n_points}  {rmse}{warn}")
print(
    f"\nViolations calendaires : {len(surface_result.diagnostics.calendar_violations)} "
    f"(nécessite ≥ 2 échéances)"
)

Slices SVI calibrées : 1
   27j  [SVI  ]  n=52  RMSE=0.00167

Violations calendaires : 0 (nécessite ≥ 2 échéances)


In [10]:
slices = surface_result.parameters.slices
raw_pts = surface_result.raw_points
if not slices:
    print("Aucune slice — plot ignoré.")
else:
    ncols = min(3, len(slices))
    nrows = (len(slices) + ncols - 1) // ncols
    dte_labels = [f"{int(sl.maturity_years * 365)}j  {sl.method.upper()}" for sl in slices]
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        shared_yaxes=True,
        horizontal_spacing=0.06,
        vertical_spacing=0.12,
        subplot_titles=dte_labels,
    )
    for idx, sl in enumerate(slices):
        row, col = idx // ncols + 1, idx % ncols + 1
        dte = int(sl.maturity_years * 365)
        raw = next((r for r in raw_pts if abs(r.maturity_years - sl.maturity_years) < 1e-4), None)
        if raw and raw.log_moneyness:
            fig.add_trace(
                go.Scatter(
                    x=raw.log_moneyness,
                    y=raw.total_variance,
                    mode="markers",
                    marker=dict(
                        color=C["blue"],
                        size=7,
                        opacity=0.85,
                        line=dict(color=C["white"], width=0.5),
                    ),
                    name="Marché" if idx == 0 else None,
                    showlegend=(idx == 0),
                    hovertemplate=f"<b>{dte}j Marché</b><br>k: %{{x:.3f}}<br>w(k): %{{y:.5f}}<extra></extra>",
                ),
                row=row,
                col=col,
            )
        if sl.method == "svi" and sl.params is not None:
            p = sl.params
            k_grid = np.linspace(-0.5, 0.5, 200)
            tv_fit = svi_total_variance(k_grid, a=p.a, b=p.b, rho=p.rho, m=p.m, sigma=p.sigma)
            fig.add_trace(
                go.Scatter(
                    x=k_grid,
                    y=tv_fit,
                    mode="lines",
                    line=dict(color=C["red"], width=2),
                    name="SVI fit" if idx == 0 else None,
                    showlegend=(idx == 0),
                    hovertemplate=f"<b>{dte}j SVI</b><br>k: %{{x:.3f}}<br>ŵ(k): %{{y:.5f}}<extra></extra>",
                ),
                row=row,
                col=col,
            )
    fig.update_yaxes(tickformat=".4f")
    fig.update_layout(
        title=f"{SYMBOL} — Calibration SVI : variance totale vs log-moneyness", height=300 * nrows
    )
    fig.show()

In [11]:
grid = surface_result.grid
n_mat = len(grid.maturity_years) if grid is not None else 0
if grid is None or n_mat < 2:
    print(f"Surface 3D non tracée : {n_mat} maturité(s) dans la grille.")
    print(
        "→ Fournis un échantillon multi-échéances : surface 3D + term structure + violations "
        "calendaires se peuplent alors automatiquement (cette cellule, telle quelle)."
    )
else:
    k_axis, t_axis = np.array(grid.log_moneyness), np.array(grid.maturity_years)
    tv = np.array(grid.total_variance)
    with np.errstate(divide="ignore", invalid="ignore"):
        iv_grid = np.where(
            t_axis[:, None] > 0, np.sqrt(np.maximum(tv, 0) / t_axis[:, None]), np.nan
        )
    kk, tt = np.meshgrid(k_axis, t_axis)
    fig = go.Figure()
    fig.add_trace(
        go.Surface(
            x=kk,
            y=tt,
            z=iv_grid,
            colorscale=SURFACE_COLORSCALE,
            opacity=0.88,
            lighting=dict(ambient=0.7, diffuse=0.6, roughness=0.5, specular=0.1),
            lightposition=dict(x=1000, y=1000, z=1000),
            contours=dict(
                z=dict(show=True, usecolormap=True, highlightcolor=C["white"], project_z=True),
                x=dict(show=False),
                y=dict(show=False),
            ),
            colorbar=dict(
                title=dict(text="IV", side="right"), tickformat=".0%", thickness=16, len=0.7, x=1.02
            ),
            hovertemplate="log-m: %{x:.3f}<br>Maturité: %{y:.3f}an<br>IV: %{z:.1%}<extra></extra>",
            name="IV Surface",
        )
    )
    viol = surface_result.diagnostics.calendar_violations
    if viol:
        fig.add_trace(
            go.Scatter3d(
                x=[v.log_moneyness for v in viol],
                y=[v.maturity_high for v in viol],
                z=[np.sqrt(max(v.variance_high, 0.0) / v.maturity_high) for v in viol],
                mode="markers",
                marker=dict(
                    color=C["red"], size=6, symbol="x", line=dict(color=C["white"], width=1)
                ),
                hovertemplate="<b>VIOLATION CALENDAIRE</b><br>log-m: %{x:.3f}<br>Maturité: %{y:.3f}an<br>IV: %{z:.1%}<extra></extra>",
                name="Violation calendaire",
            )
        )
    fig.update_layout(
        title=f"{SYMBOL} — Surface de volatilité implicite (SVI)",
        height=640,
        scene=dict(
            xaxis=dict(
                title="Log-Moneyness",
                gridcolor=C["slate400"],
                backgroundcolor=C["slate100"],
                showbackground=True,
            ),
            yaxis=dict(
                title="Maturité (an)",
                gridcolor=C["slate400"],
                backgroundcolor=C["slate100"],
                showbackground=True,
            ),
            zaxis=dict(
                title="IV",
                tickformat=".0%",
                gridcolor=C["slate400"],
                backgroundcolor=C["slate100"],
                showbackground=True,
            ),
            camera=SURFACE_CAMERA,
            aspectmode="manual",
            aspectratio=SURFACE_ASPECT,
        ),
        margin=dict(l=0, r=0, t=56, b=0),
    )
    fig.show()
    print(
        f"✓ fig.to_json() OK — {len(fig.to_json().encode()) / 1024:.1f} KB (prêt FastAPI → react-plotly.js)"
    )

Surface 3D non tracée : 1 maturité(s) dans la grille.
→ Fournis un échantillon multi-échéances : surface 3D + term structure + violations calendaires se peuplent alors automatiquement (cette cellule, telle quelle).


## 5 · Pricing & Greeks
Repricing Black-76 + Greeks analytiques, vol échantillonnée sur la surface SVI calibrée.

In [12]:
pricing_params = PricingParams.from_config(load_config(CONFIGS / "pricing.yaml"))
solved_by_exp: dict = defaultdict(list)
for p in iv_points:
    if p.status == "solved" and p.implied_vol is not None:
        solved_by_exp[p.expiry_date].append(p)

_grid = surface_result.grid


def _surface_vol(k: float, t_yr: float, fallback: float) -> float:
    if _grid is None or t_yr <= 0:
        return fallback
    tv = _grid.total_variance_at(maturity=t_yr, k=k)
    return float(np.sqrt(max(tv, 0.0) / t_yr))


pricing_inputs: dict[str, PricingInput] = {}
pricing_vol: dict[str, float] = {}
results: dict = {}
first_exp = None
spot = float(snapshot.reference_spot)
T = 0.0
if not solved_by_exp:
    print("Aucun IvPoint solved — pricing impossible.")
else:
    first_exp = sorted(solved_by_exp)[0]
    pts_exp = solved_by_exp[first_exp]
    fwd_mat = next((m for m in forward_curve.maturities if m.expiry == first_exp), None)
    rate = float(fwd_mat.rate) if fwd_mat else 0.0
    carry = float(fwd_mat.implied_carry) if fwd_mat else 0.0
    forward = float(fwd_mat.forward_price) if fwd_mat else spot
    T = pts_exp[0].maturity_years
    for p in pts_exp:
        c = parse_instrument_key(p.contract_key)
        if not isinstance(c, OptionContract):
            continue
        vol = _surface_vol(p.log_moneyness, T, p.implied_vol)
        pricing_vol[p.contract_key] = vol
        pricing_inputs[p.contract_key] = PricingInput(
            spot=spot,
            strike=float(c.strike),
            maturity=T,
            vol=vol,
            rate=rate,
            carry=carry,
            is_call=(c.right == Right.CALL),
            multiplier=c.multiplier,
            style=OptionStyle.EUROPEAN,
        )
    results = {key: price(inp, pricing_params) for key, inp in pricing_inputs.items()}
    print(f"Échéance pricée : {first_exp} ({int(T * 365)}j)  —  {len(results)} options")
    print(f"Spot=€{spot:,.2f}  Forward=€{forward:,.2f}  T={T:.4f}y  (vol depuis surface SVI)")

Échéance pricée : 2026-07-01 (27j)  —  52 options
Spot=€1,454.90  Forward=€1,455.95  T=0.0740y  (vol depuis surface SVI)


In [13]:
if not results:
    print("Pas de données — plot ignoré.")
else:
    calls_data = sorted(
        [
            (parse_instrument_key(k), results[k])
            for k in results
            if isinstance(parse_instrument_key(k), OptionContract)
            and parse_instrument_key(k).right == Right.CALL
        ],
        key=lambda x: float(x[0].strike),
    )
    puts_data = sorted(
        [
            (parse_instrument_key(k), results[k])
            for k in results
            if isinstance(parse_instrument_key(k), OptionContract)
            and parse_instrument_key(k).right == Right.PUT
        ],
        key=lambda x: float(x[0].strike),
    )
    greeks_cfg = [
        ("price", "Prix (EUR)", ".2f", C["blue"], C["teal"], (1, 1)),
        ("delta", "Delta Δ", ".3f", C["blue"], C["teal"], (1, 2)),
        ("gamma", "Gamma Γ", ".5f", C["violet"], C["amber"], (2, 1)),
        ("vega", "Vega", ".2f", C["violet"], C["amber"], (2, 2)),
    ]
    fig = make_subplots(
        rows=2,
        cols=2,
        shared_xaxes=True,
        vertical_spacing=0.08,
        horizontal_spacing=0.08,
        subplot_titles=[g[1] for g in greeks_cfg],
    )
    for field, label, fmt, c_call, c_put, (row, col) in greeks_cfg:
        for side, data, color, dash in [
            ("Call", calls_data, c_call, "solid"),
            ("Put", puts_data, c_put, "dot"),
        ]:
            fig.add_trace(
                go.Scatter(
                    x=[float(c.strike) for c, _ in data],
                    y=[getattr(r, field) for _, r in data],
                    mode="lines",
                    line=dict(color=color, width=2, dash=dash),
                    name=side,
                    legendgroup=side,
                    showlegend=(field == "price"),
                    hovertemplate=f"Strike %{{x:.0f}}<br>{label}: %{{y:{fmt}}}<extra></extra>",
                ),
                row=row,
                col=col,
            )
        fig.update_yaxes(tickformat=fmt, row=row, col=col)
    fig.add_vline(x=spot, line_dash="dash", line_color=C["slate400"])
    fig.update_xaxes(title_text="Strike (EUR)", row=2, col=1)
    fig.update_xaxes(title_text="Strike (EUR)", row=2, col=2)
    fig.update_layout(
        title=f"{SYMBOL} {first_exp} ({int(T * 365)}j) — Greeks vs strike (call plein, put pointillé)",
        height=560,
    )
    fig.show()

## 6 · 🎁 Moteur vs broker Saxo — validation… et détection d'incohérence
Impossible chez Deribit (pas de Greeks fournis). Saxo publie `mark_iv` + delta/gamma/vega/theta :
on confronte notre moteur (qui **recalcule tout depuis les quotes brutes**) à ces champs. Sur cet
échantillon delayed : le **delta** moteur ≈ broker (sain), mais le **`mark_iv`** broker ne colle
ni au mid coté (≈ via notre solveur) ni à son **propre delta** → champ **incohérent** que le
moteur attrape en repartant des prix. C'est précisément l'intérêt de tout reconstruire.

In [14]:
broker: dict[str, dict[str, float]] = defaultdict(dict)
for e in events:
    if e.field_name in ("delta", "gamma", "vega", "theta", "mark_iv") and e.field_value is not None:
        broker[e.instrument_key][e.field_name] = float(e.field_value)

if not results or not solved_by_exp:
    print("Pas de pricing — comparaison ignorée.")
else:
    iv_eng, iv_brk, d_eng, d_brk = [], [], [], []
    for p in solved_by_exp[first_exp]:
        b, r = broker.get(p.contract_key), results.get(p.contract_key)
        if not b or not r:
            continue
        if "mark_iv" in b and p.implied_vol is not None:
            iv_eng.append(p.implied_vol * 100)
            iv_brk.append(b["mark_iv"] * 100)
        if "delta" in b:
            d_eng.append(r.delta)
            d_brk.append(b["delta"])
    if iv_eng:
        err = np.array(iv_eng) - np.array(iv_brk)
        print(
            f"IV moteur vs mark_iv broker : n={len(iv_eng)}  écart moyen={err.mean():+.2f}pt  "
            f"|écart| médian={np.median(np.abs(err)):.2f}pt"
        )
    if d_eng:
        derr = np.array(d_eng) - np.array(d_brk)
        print(
            f"Delta moteur vs broker      : n={len(d_eng)}  écart moyen={derr.mean():+.4f}  "
            f"|écart| médian={np.median(np.abs(derr)):.4f}"
        )
    print("  -> delta broker cohérent avec le moteur ; mark_iv broker incohérent (ni mid, ni son")
    print("     propre delta) : champ delayed à ne pas croire — le moteur recalcule juste.")

    fig = make_subplots(
        cols=2, subplot_titles=("IV : moteur vs mark_iv broker (%)", "Delta : moteur vs broker")
    )
    if iv_eng:
        lo, hi = min(iv_eng + iv_brk), max(iv_eng + iv_brk)
        fig.add_trace(
            go.Scatter(
                x=iv_brk,
                y=iv_eng,
                mode="markers",
                marker=dict(
                    color=C["blue"], size=7, opacity=0.8, line=dict(color=C["white"], width=0.5)
                ),
                name="IV",
                showlegend=False,
                hovertemplate="broker: %{x:.1f}%<br>moteur: %{y:.1f}%<extra></extra>",
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=[lo, hi],
                y=[lo, hi],
                mode="lines",
                line=dict(color=C["slate400"], dash="dash", width=1),
                name="moteur = broker",
                showlegend=False,
            ),
            row=1,
            col=1,
        )
    if d_eng:
        lo, hi = min(d_eng + d_brk), max(d_eng + d_brk)
        fig.add_trace(
            go.Scatter(
                x=d_brk,
                y=d_eng,
                mode="markers",
                marker=dict(
                    color=C["green"], size=7, opacity=0.8, line=dict(color=C["white"], width=0.5)
                ),
                name="Delta",
                showlegend=False,
                hovertemplate="broker: %{x:.3f}<br>moteur: %{y:.3f}<extra></extra>",
            ),
            row=1,
            col=2,
        )
        fig.add_trace(
            go.Scatter(
                x=[lo, hi],
                y=[lo, hi],
                mode="lines",
                line=dict(color=C["slate400"], dash="dash", width=1),
                name="moteur = broker",
                showlegend=False,
            ),
            row=1,
            col=2,
        )
    fig.update_xaxes(title_text="broker", row=1, col=1)
    fig.update_xaxes(title_text="broker", row=1, col=2)
    fig.update_yaxes(title_text="moteur", row=1, col=1)
    fig.update_yaxes(title_text="moteur", row=1, col=2)
    fig.update_layout(
        title=f"{SYMBOL} — Moteur vs broker : delta cohérent, mark_iv broker incohérent", height=420
    )
    fig.show()

IV moteur vs mark_iv broker : n=52  écart moyen=-15.35pt  |écart| médian=12.92pt
Delta moteur vs broker      : n=52  écart moyen=-0.0005  |écart| médian=0.0029
  -> delta broker cohérent avec le moteur ; mark_iv broker incohérent (ni mid, ni son
     propre delta) : champ delayed à ne pas croire — le moteur recalcule juste.


## 7 · Risque — straddle ATM fictif
Un straddle ATM (call + put au **même** strike). Delta ≈ 0, long Gamma / long Vega / short Theta.

In [15]:
risk_lines: list = []
agg = None
positions = None
atm_call_key = atm_put_key = None
theta_suffix = "/j"
if not results:
    print("Pas de pricing — risk ignoré.")
else:
    lm = {p.contract_key: p.log_moneyness for p in solved_by_exp[first_exp]}
    atm_key = min(results, key=lambda k: abs(lm.get(k, 999.0)))
    atm_strike = parse_instrument_key(atm_key).strike

    def _leg(right: Right) -> str | None:
        for k in results:
            c = parse_instrument_key(k)
            if c.right == right and c.strike == atm_strike:
                return k
        return None

    atm_call_key, atm_put_key = _leg(Right.CALL), _leg(Right.PUT)
    if not atm_call_key or not atm_put_key:
        print("Call ou put ATM introuvable au même strike — risk ignoré.")
    else:
        positions = PositionSet(
            positions=(
                Position(instrument_key=atm_call_key, quantity=Decimal("1")),
                Position(instrument_key=atm_put_key, quantity=Decimal("1")),
            ),
            source="notebook-saxo-demo",
            source_ts=AS_OF,
        )
        analytics = {
            key: InstrumentAnalytics(
                pricing=results[key], multiplier=parse_instrument_key(key).multiplier
            )
            for key in (atm_call_key, atm_put_key)
        }
        risk_lines = compute_risk_lines(positions, analytics)
        agg = aggregate_risk(risk_lines, by_underlying)[0]
        theta_suffix = "/an" if pricing_params.theta_unit == "year" else "/j"
        print(f"Strike ATM : K=€{float(atm_strike):,.0f}")
        print(
            f"Valeur=€{agg.value:.2f}  Delta={agg.delta:+.4f} (≈0)  "
            f"Vega=€{agg.dollar_vega:+.2f}  Theta=€{agg.theta:+.2f}{theta_suffix}"
        )

Strike ATM : K=€1,460
Valeur=€11524.48  Delta=+1.7050 (≈0)  Vega=€+31647.21  Theta=€-78087.80/an


In [16]:
if not risk_lines or agg is None:
    print("Pas de données risk — plot ignoré.")
else:
    leg_labels = [
        f"{parse_instrument_key(li.instrument_key).right.value} "
        f"K={float(parse_instrument_key(li.instrument_key).strike):,.0f}"
        for li in risk_lines
    ]
    leg_labels.append("NET")
    greeks_risk = {
        "Delta": [li.delta for li in risk_lines] + [agg.delta],
        "Gamma": [li.gamma for li in risk_lines] + [agg.gamma],
        "Vega": [li.dollar_vega for li in risk_lines] + [agg.dollar_vega],
        "Theta": [li.theta for li in risk_lines] + [agg.theta],
    }
    greek_colors = {
        "Delta": C["blue"],
        "Gamma": C["violet"],
        "Vega": C["teal"],
        "Theta": C["amber"],
    }
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=list(greeks_risk.keys()),
        horizontal_spacing=0.12,
        vertical_spacing=0.15,
    )
    for (greek, vals), (row, col) in zip(
        greeks_risk.items(), [(1, 1), (1, 2), (2, 1), (2, 2)], strict=False
    ):
        fig.add_trace(
            go.Bar(
                y=leg_labels,
                x=vals,
                orientation="h",
                marker_color=[C["red"] if v < 0 else greek_colors[greek] for v in vals],
                name=greek,
                showlegend=False,
                hovertemplate=f"<b>%{{y}}</b><br>{greek}: %{{x:.4f}}<extra></extra>",
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(
            zeroline=True, zerolinecolor=C["slate400"], zerolinewidth=1.5, row=row, col=col
        )
    fig.update_layout(
        title=f"{SYMBOL} straddle ATM — Greeks par jambe (rouge = négatif)", height=460
    )
    fig.show()

## 8 · Scénarios de stress (grille actions)
Chocs calibrés **actions** (plus serrés que le crypto) : spot ±5/10 %, vol ±5 pts, passage du
temps. `full` = repricing complet (vérité) ; `local` = approximation Greeks (Eq. 19) ; l'écart =
non-linéarités (gamma/vega).

In [17]:
scenario_results = []
if not risk_lines:
    print("Pas de données — scénarios ignorés.")
else:
    scenario_grid = ScenarioGrid(
        scenarios=(
            Scenario(
                name="spot_up_5", family="spot", spot_shock=0.05, vol_shock=0.0, time_shock_days=0
            ),
            Scenario(
                name="spot_up_10", family="spot", spot_shock=0.10, vol_shock=0.0, time_shock_days=0
            ),
            Scenario(
                name="spot_down_5",
                family="spot",
                spot_shock=-0.05,
                vol_shock=0.0,
                time_shock_days=0,
            ),
            Scenario(
                name="spot_down_10",
                family="spot",
                spot_shock=-0.10,
                vol_shock=0.0,
                time_shock_days=0,
            ),
            Scenario(
                name="vol_up_5pt", family="vol", spot_shock=0.0, vol_shock=0.05, time_shock_days=0
            ),
            Scenario(
                name="vol_down_5pt",
                family="vol",
                spot_shock=0.0,
                vol_shock=-0.05,
                time_shock_days=0,
            ),
            Scenario(
                name="selloff_d10_v10",
                family="crash",
                spot_shock=-0.10,
                vol_shock=0.10,
                time_shock_days=0,
            ),
            Scenario(
                name="rally_u10_v_down",
                family="rally",
                spot_shock=0.10,
                vol_shock=-0.05,
                time_shock_days=0,
            ),
            Scenario(
                name="roll_1d", family="time", spot_shock=0.0, vol_shock=0.0, time_shock_days=1
            ),
            Scenario(
                name="roll_7d", family="time", spot_shock=0.0, vol_shock=0.0, time_shock_days=7
            ),
        ),
        version="notebook-saxo-v1",
    )
    states = {
        key: pricing_inputs[key] for key in (atm_call_key, atm_put_key) if key in pricing_inputs
    }
    scenario_results = run_grid(positions, states, pricing_params, scenario_grid)
    print(f"Scénarios exécutés : {len(scenario_results)}")
    for r in scenario_results:
        print(
            f"  {r.scenario:<22} full=€{r.full_pnl:>+9.2f}  local=€{r.local_pnl:>+9.2f}  "
            f"Δ=€{r.full_pnl - r.local_pnl:>+8.2f}"
        )

Scénarios exécutés : 10
  spot_up_5              full=€ +1537.28  local=€ +1594.94  Δ=€  -57.66
  spot_up_10             full=€ +5502.51  local=€ +6131.71  Δ=€ -629.20
  spot_down_5            full=€ +1344.87  local=€ +1346.88  Δ=€   -2.01
  spot_down_10           full=€ +5371.37  local=€ +5635.58  Δ=€ -264.21
  vol_up_5pt             full=€ +1582.16  local=€ +1582.36  Δ=€   -0.20
  vol_down_5pt           full=€ -1582.51  local=€ -1582.36  Δ=€   -0.15
  selloff_d10_v10        full=€ +7253.77  local=€ +8800.30  Δ=€-1546.53
  rally_u10_v_down       full=€ +4502.05  local=€ +4549.35  Δ=€  -47.29
  roll_1d                full=€  -215.93  local=€  -213.94  Δ=€   -1.99
  roll_7d                full=€ -1608.37  local=€ -1497.57  Δ=€ -110.80


In [18]:
if not scenario_results:
    print("Pas de scénarios — plot ignoré.")
else:
    names = [r.scenario for r in scenario_results]
    full = np.array([r.full_pnl for r in scenario_results])
    local = np.array([r.local_pnl for r in scenario_results])
    abs_err = np.abs(full - local)
    fig = make_subplots(
        rows=1,
        cols=2,
        column_widths=[0.5, 0.5],
        subplot_titles=("Full reprice vs approx Greeks", "PnL par scénario"),
    )
    fig.add_trace(
        go.Scatter(
            x=full,
            y=local,
            mode="markers",
            marker=dict(
                color=abs_err,
                colorscale=[[0, C["green"]], [0.5, C["amber"]], [1, C["red"]]],
                colorbar=dict(title="Err abs", x=0.47, thickness=12, len=0.8),
                size=9,
                opacity=0.9,
                line=dict(color=C["white"], width=0.5),
            ),
            customdata=names,
            hovertemplate="<b>%{customdata}</b><br>Full: %{x:.2f}<br>Greeks: %{y:.2f}<br>Err: %{marker.color:.2f}<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=1,
    )
    rng = [float(min(full.min(), local.min())), float(max(full.max(), local.max()))]
    fig.add_trace(
        go.Scatter(
            x=rng,
            y=rng,
            mode="lines",
            line=dict(color=C["slate400"], dash="dash", width=1),
            name="full=local",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=names,
            y=full,
            marker_color=[C["green"] if v >= 0 else C["red"] for v in full],
            name="Full reprice",
            hovertemplate="<b>%{x}</b><br>Full: %{y:.2f}€<extra></extra>",
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Bar(
            x=names,
            y=local,
            marker_color=C["blue"],
            opacity=0.6,
            marker_pattern_shape="/",
            name="Approx Greeks",
            hovertemplate="<b>%{x}</b><br>Greeks: %{y:.2f}€<extra></extra>",
        ),
        row=1,
        col=2,
    )
    fig.update_xaxes(title_text="Full reprice PnL", row=1, col=1)
    fig.update_yaxes(title_text="Approx Greeks PnL", row=1, col=1)
    fig.update_xaxes(tickangle=30, row=1, col=2)
    fig.update_yaxes(title_text="PnL (EUR)", row=1, col=2)
    fig.update_layout(
        title=f"{SYMBOL} straddle ATM — Analyse PnL de stress (grille actions)",
        barmode="group",
        height=460,
    )
    fig.show()

## Récapitulatif de la pile

| Étape | Sortie | Module moteur |
|---|---|---|
| Échantillon committé | events (offline) | `storage.events_from_json` |
| Snapshot + QC | état filtré, raisons de rejet | `snapshots` / `qc` |
| Forward | parité put-call, taux implicite | `forwards` |
| Inversion IV | smile (skew actions réel) | `iv` |
| Surface SVI | slice(s) calibrée(s), surface 3D si ≥2 maturités | `surfaces` |
| Pricing & Greeks | Black-76 analytique | `pricing` |
| Moteur vs broker | delta validé, incohérence `mark_iv` détectée | — |
| Risque + scénarios | straddle, stress actions | `risk` |

Reconstruction **déterministe** depuis une donnée réelle committée — mêmes inputs → mêmes
sorties, sans broker ni réseau. Surface 3D : fournir un échantillon multi-échéances et les
cellules surface/term-structure/violations se peuplent automatiquement.